# 02 · Descomposición STL de los 10 productos con más ventas (5.3.7)

**Objetivo.** Comprobar, con un segundo método, la tendencia y la estacionalidad que 5.3.5 describió con sumas móviles e índices, ahora **producto por producto** en los 10 productos con más piezas vendidas. Estos 10 productos son el conjunto sobre el que se modelará (5.5). El análisis es descriptivo: no ajusta modelos de proyección ni modifica la base de datos.

**Cómo ejecutarlo.**
1. Desde la raíz del repositorio, con el entorno virtual activo y el archivo `.env` con `DATABASE_URL` (el mismo que usa el ETL).
2. Ejecutar las celdas en orden. Las figuras se guardan en `notebooks/figures/` con el prefijo `stl_` (esa regla está en `.gitignore`: no se versionan).

**Contenido.** 1 Conceptos · 2 Configuración · 3 Carga y verificación · 4 Selección del top 10 · 5 Descomposición · 6 Fuerza, mes pico y valle · 7 Comparación con 5.3.5 · 8 Validaciones · 9 Aditivo frente a logaritmos · 10 Atípicos · 11 Resumen y decisiones para 5.5.

## 1. Conceptos que se usan en este notebook

**Serie de tiempo mensual.** Las piezas vendidas de un producto en cada mes, de enero de 2022 a septiembre de 2026 (57 observaciones).

**Tendencia (T).** El nivel de fondo de las ventas, que sube o baja lentamente una vez que se quitan el efecto del calendario y el ruido de cada mes. Responde: ¿este producto vende hoy más o menos que hace dos años, sin importar si es diciembre o septiembre?

**Estacionalidad (S).** Un patrón que se repite cada año en los mismos meses (por ejemplo, un diciembre alto y un septiembre bajo). Se expresa en piezas por encima o por debajo de la tendencia: un valor de +120 en diciembre significa que diciembre vende, típicamente, 120 piezas más de lo que marca la tendencia.

**Residuo (R).** Lo que no explican ni la tendencia ni la estacionalidad: variaciones aleatorias, pedidos extraordinarios, faltantes de stock. Un residuo grande en un mes es una señal de que ese mes fue inusual.

**STL** (*Seasonal and Trend decomposition using Loess*; Cleveland et al., 1990). Separa la serie en esos tres componentes mediante suavizados locales (*loess*). Se eligió porque funciona con cualquier tipo de estacionalidad, deja que el patrón estacional cambie lentamente entre años y puede ser robusto a valores extremos.

**Modelo aditivo.** Se asume que, mes a mes:

$$y_t = T_t + S_t + R_t$$

es decir, el efecto estacional es una cantidad fija de piezas sobre la tendencia. Es la versión más fácil de interpretar y la que se usa como principal. La alternativa es el **modelo multiplicativo** ($y_t = T_t \times S_t \times R_t$), donde el efecto es un porcentaje de la tendencia; se obtiene aplicando logaritmo a la serie, y se compara en la sección 9.

**Parámetros de STL.**
- `period = 12`: la estacionalidad se repite cada 12 meses.
- `seasonal = 7`: longitud del suavizado estacional (siempre impar). Indica cuántos años de un mismo mes se usan para estimar su efecto. Un valor grande da un patrón casi fijo entre años; uno pequeño deja que cambie más. Cada mes tiene solo 4 o 5 observaciones (una por año), por lo que 7 usa prácticamente todas.
- `trend`: ventana (en meses) del suavizado de la tendencia. Se deja el valor por defecto de `statsmodels`, que se calcula a partir de `period` y `seasonal` (alrededor de 23 meses): una ventana grande produce una tendencia lenta, que suaviza los cambios bruscos.
- `robust = True`: los meses con residuo muy grande pesan menos en el ajuste, de modo que un pico aislado no distorsiona la tendencia ni el patrón estacional. El pico sigue apareciendo en el residuo.

**Fuerza de la tendencia ($F_T$) y de la estacionalidad ($F_S$)** (Hyndman y Athanasopoulos, 2021). Miden qué tan dominante es cada componente frente al residuo, con valores entre 0 y 1:

$$F_T = \max\left(0,\ 1 - \frac{\mathrm{Var}(R)}{\mathrm{Var}(T+R)}\right) \qquad F_S = \max\left(0,\ 1 - \frac{\mathrm{Var}(R)}{\mathrm{Var}(S+R)}\right)$$

Si el residuo es pequeño frente al componente, la fuerza se acerca a 1 (el componente explica casi toda la variación). Si el residuo es del mismo tamaño que el componente o mayor, se acerca a 0 (el componente casi no se distingue del ruido). No hay un umbral oficial; se usan para comparar productos entre sí. *Ejemplo:* si $\mathrm{Var}(R) = 10$ y $\mathrm{Var}(S+R) = 40$, entonces $F_S = 1 - 10/40 = 0.75$.

**Mes pico y mes valle.** Se promedia el componente estacional de cada mes del calendario a lo largo de los años. El **mes pico** es el de mayor promedio y el **mes valle** el de menor. Se expresan como porcentaje de la tendencia media del producto, para poder compararlos entre productos de distinto tamaño.

**Índice estacional comparable con 5.3.5.** $1 + S_{\text{mes}} / \bar{T}$ (efecto del mes dividido entre la tendencia media). Así, 1.13 significa que ese mes vende, típicamente, 13% más que el nivel de fondo, igual que el índice de 5.3.5.

**Validaciones.** V1 integridad de los datos; V2 identidad de la descomposición ($T+S+R$ = serie original); V3 el residuo no debe tener patrón (autocorrelación a 1 y a 12 meses dentro de la banda aproximada $\pm 1.96/\sqrt{n}$); V4 sensibilidad: los resultados no deben cambiar mucho al mover los ajustes.

## 2. Configuración

In [ ]:
import sys
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

warnings.filterwarnings("ignore", category=FutureWarning)

# Raíz del proyecto (para importar src/ aunque el notebook se abra desde notebooks/)
RAIZ = Path.cwd().resolve()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

# --- Datos ---
PRIMER_MES = date(2022, 1, 1)
ULTIMO_MES = date(2026, 9, 1)          # último mes cerrado (corte de 5.3.1)
TOP_N = 10

# --- STL (se explican en la sección 1) ---
PERIODO = 12
SEASONAL = 7
ROBUSTO = True
VARIANTES = {                           # sensibilidad (V4)
    "principal (seasonal=7, robusto)": dict(seasonal=7, robust=True),
    "seasonal=3": dict(seasonal=3, robust=True),
    "no robusto": dict(seasonal=7, robust=False),
}

# --- Referencias de 5.3 ---
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
TEMPORADA_ALTA = {3, 4, 5, 7, 8, 11, 12}   # clasificación observada, 5.3.5
INDICE_535 = pd.Series([0.97, 0.99, 1.01, 1.06, 1.14, 0.97, 1.01, 1.07, 0.75, 0.88, 1.02, 1.13],
                       index=range(1, 13))  # índice estacional ajustado, 5.3.5
PEDIDO_CONFIRMADO = ("8419", "2022-12-01")  # pedido del kit confirmado por la empresa (5.3.4)

# --- Figuras (no se versionan: regla notebooks/figures/stl_* en .gitignore) ---
CARPETA_FIGURAS = RAIZ / "notebooks" / "figures"
CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)
AZUL, GRIS, NARANJA = "#1f5fa8", "#8a8f98", "#d9822b"


def guardar(fig, nombre):
    fig.savefig(CARPETA_FIGURAS / f"stl_{nombre}.png", dpi=200, bbox_inches="tight")


plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 3. Carga desde Supabase y verificación (V1)

Se leen las tablas `productos` y `ventas` con el repositorio de `src/db` (solo lectura). La conexión usa `DATABASE_URL` del archivo `.env`. Se cargan los datos completos del ETL; el filtro al top 10 se hace en la sección 4.

In [ ]:
from src.db.repository import Repositorio, conectar

if "repo" not in globals():            # permite inyectar un repositorio de prueba
    repo = Repositorio(conectar())

productos = repo.productos()
ventas = repo.ventas(PRIMER_MES, ULTIMO_MES)
if hasattr(repo, "conn"):
    repo.conn.close()                  # solo se leía

ventas["fecha"] = pd.to_datetime(ventas["fecha"])
meses = pd.date_range(PRIMER_MES, ULTIMO_MES, freq="MS")

# Matriz mes × producto (los productos de `ventas` son los de historial completo)
matriz = ventas.pivot(index="fecha", columns="clave", values="cantidad").reindex(meses)
matriz.index.freq = "MS"

# V1: integridad de los datos
v1 = {
    "productos en la tabla productos": len(productos),
    "productos con historial completo": int(productos["historial_completo"].sum()),
    "productos con ventas cargadas": matriz.shape[1],
    "meses por producto (esperado %d)" % len(meses): int(matriz.notna().sum().min()) if matriz.notna().sum().nunique() == 1 else "distinto",
    "celdas vacías": int(matriz.isna().sum().sum()),
    "valores negativos": int((matriz < 0).sum().sum()),
    "fechas que no son día 01": int((ventas["fecha"].dt.day != 1).sum()),
    "filas de ventas": len(ventas),
}
display(pd.Series(v1, name="valor").to_frame())
assert v1["celdas vacías"] == 0 and v1["valores negativos"] == 0
assert v1["productos con ventas cargadas"] == v1["productos con historial completo"]
validaciones = {}

## 4. Selección del top 10

**Criterio.** Los 10 productos con más piezas vendidas en todo el historial (enero de 2022 a septiembre de 2026), entre los de historial completo y con unidad en piezas. Los productos en kilogramos no se comparan con piezas y quedan fuera.

Además se compara con el top 10 de los últimos 12 meses (el criterio de 5.3.6): un producto que lideró los primeros años pero hoy vende poco sería una mala elección para proyectar.

In [ ]:
unidad = productos.set_index("clave")["unidad"]
descripcion = productos.set_index("clave")["descripcion"]

claves_pza = [c for c in matriz.columns if unidad[c] == "PZA"]
print(f"Productos con historial completo: {matriz.shape[1]}  |  en piezas: {len(claves_pza)}  |  en kg (excluidos): {matriz.shape[1] - len(claves_pza)}")

total_pza = matriz[claves_pza].sum()
top = total_pza.nlargest(TOP_N)
claves_top = list(top.index)

ult12 = matriz[claves_pza].iloc[-12:].sum()
rank12 = ult12.rank(ascending=False, method="min").astype(int)

tabla_top = pd.DataFrame({
    "clave": claves_top,
    "descripcion": [descripcion[c] for c in claves_top],
    "piezas_totales": top.values,
    "% del total en piezas": (top / total_pza.sum() * 100).round(1).values,
    "% acumulado": (top.cumsum() / total_pza.sum() * 100).round(1).values,
    "lugar en últimos 12 meses": [rank12[c] for c in claves_top],
})
tabla_top.index = range(1, TOP_N + 1)
display(tabla_top.style.format({"piezas_totales": "{:,.0f}"}))

print(f"\nLos {TOP_N} productos concentran {top.sum() / total_pza.sum() * 100:.1f}% de las piezas de todo el historial "
      f"y {ult12[claves_top].sum() / ult12.sum() * 100:.1f}% de las piezas de los últimos 12 meses.")
top12 = set(ult12.nlargest(TOP_N).index)
print(f"Coinciden con el top {TOP_N} de los últimos 12 meses: {len(top12 & set(claves_top))} de {TOP_N}.")
fuera = [c for c in claves_top if c not in top12]
if fuera:
    print("Fuera del top 10 reciente:", ", ".join(f"{c} (lugar {rank12[c]})" for c in fuera))

## 5. Descomposición STL (modelo aditivo)

Para cada producto se aplica `STL(serie, period=12, seasonal=7, robust=True)`. Cada figura muestra, de arriba abajo, la serie observada, la tendencia, el componente estacional y el residuo. En el residuo se marcan los meses que superan 3 desviaciones robustas (ver sección 10).

In [ ]:
def descomponer(serie, seasonal=SEASONAL, robust=ROBUSTO):
    """STL aditivo de una serie mensual. Devuelve observado, tendencia, estacional y residuo."""
    r = STL(serie, period=PERIODO, seasonal=seasonal, robust=robust).fit()
    return pd.DataFrame({"observado": serie, "tendencia": r.trend, "estacional": r.seasonal, "residuo": r.resid})


def graficar(d, clave, titulo=None):
    fig, ejes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
    for ax, col, color in zip(ejes, ["observado", "tendencia", "estacional", "residuo"], [AZUL, AZUL, NARANJA, GRIS]):
        ax.plot(d.index, d[col], color=color, lw=1.6)
        ax.set_ylabel(col.capitalize(), fontsize=9)
        ax.grid(axis="y", alpha=0.25)
    ejes[2].axhline(0, color=GRIS, lw=0.8)
    ejes[3].axhline(0, color=GRIS, lw=0.8)
    if clave == PEDIDO_CONFIRMADO[0]:
        f = pd.Timestamp(PEDIDO_CONFIRMADO[1])
        ejes[0].annotate("pedido confirmado", (f, d.loc[f, "observado"]), xytext=(8, -6), textcoords="offset points", fontsize=8)
    ejes[0].set_title(titulo or f"Producto {clave}: descomposición STL aditiva (piezas)", fontsize=11, loc="left")
    fig.tight_layout()
    return fig


desc = {c: descomponer(matriz[c].astype(float)) for c in claves_top}

for c in claves_top:
    fig = graficar(desc[c], c)
    guardar(fig, f"panel_{c}")
    plt.show()

En el producto 1014, una tendencia de ≈300 a inicios de 2022 significa que, en esa época, el producto vendía alrededor de 300 piezas al mes en promedio, ya sea que fuera diciembre o enero.

## 6. Fuerza de tendencia y estacionalidad, mes pico y mes valle

Para cada producto se calculan $F_T$ y $F_S$, el mes pico y el mes valle (con su efecto como porcentaje de la tendencia media) y la variación de la tendencia en los últimos 12 meses (tendencia del último mes frente a la de 12 meses antes).

In [ ]:
def fuerza(componente, residuo):
    return max(0.0, 1 - np.var(residuo) / np.var(componente + residuo))


def perfil_estacional(d):
    """Promedio del componente estacional por mes del calendario (12 valores, en piezas)."""
    return d["estacional"].groupby(d.index.month).mean()


filas, perfiles = [], {}
for c in claves_top:
    d = desc[c]
    p = perfil_estacional(d)
    perfiles[c] = p
    t_medio = d["tendencia"].mean()
    filas.append({
        "clave": c,
        "piezas_totales": total_pza[c],
        "F_tendencia": fuerza(d["tendencia"], d["residuo"]),
        "F_estacional": fuerza(d["estacional"], d["residuo"]),
        "mes_pico": MESES[p.idxmax() - 1],
        "efecto_pico_%": p.max() / t_medio * 100,
        "mes_valle": MESES[p.idxmin() - 1],
        "efecto_valle_%": p.min() / t_medio * 100,
        "tendencia_12m_%": (d["tendencia"].iloc[-1] / d["tendencia"].iloc[-13] - 1) * 100,
    })
metricas = pd.DataFrame(filas).set_index("clave")
display(metricas.style.format({"piezas_totales": "{:,.0f}", "F_tendencia": "{:.2f}", "F_estacional": "{:.2f}",
                               "efecto_pico_%": "{:+.1f}", "efecto_valle_%": "{:+.1f}", "tendencia_12m_%": "{:+.1f}"}))

## 7. Comparación con el índice estacional de 5.3.5

El índice de 5.3.5 se calculó sobre el total de los productos en piezas. Aquí se calcula, para cada uno de los 10, el índice equivalente a partir de STL ($1 + S_{\text{mes}} / \bar{T}$) y se compara con él: correlación entre ambos perfiles de 12 meses y número de meses en que el producto coincide con la clasificación de temporada alta y baja de 5.3.5.

In [ ]:
indice_stl = pd.DataFrame({c: 1 + perfiles[c] / desc[c]["tendencia"].mean() for c in claves_top})
indice_stl.index = MESES
alta = np.array([m in TEMPORADA_ALTA for m in range(1, 13)])

comp = pd.DataFrame({
    "correlación con 5.3.5": [np.corrcoef(indice_stl[c].values, INDICE_535.values)[0, 1] for c in claves_top],
    "meses que coinciden con alta/baja (de 12)": [int(((indice_stl[c].values > 1) == alta).sum()) for c in claves_top],
}, index=claves_top)
display(comp.style.format({"correlación con 5.3.5": "{:.2f}"}))

fig, ax = plt.subplots(figsize=(9, 4.5))
for m in range(1, 13):
    if m in TEMPORADA_ALTA:
        ax.axvspan(m - 0.5, m + 0.5, color=GRIS, alpha=0.10, lw=0)
for c in claves_top:
    ax.plot(range(1, 13), indice_stl[c].values, color=AZUL, alpha=0.35, lw=1.2)
ax.plot(range(1, 13), indice_stl.mean(axis=1).values, color=AZUL, lw=2.4, label="Promedio de los 10 (STL)")
ax.plot(range(1, 13), INDICE_535.values, color=NARANJA, lw=2.4, marker="o", label="Índice de 5.3.5")
ax.axhline(1, color=GRIS, lw=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES)
ax.set_ylabel("Índice estacional")
ax.set_title("Patrón estacional por producto (STL) frente al índice de 5.3.5; sombreado = temporada alta", fontsize=10, loc="left")
ax.legend(frameon=False)
fig.tight_layout()
guardar(fig, "patron_estacional")
plt.show()
print("Correlación entre el promedio de los 10 y el índice de 5.3.5: %.2f" % np.corrcoef(indice_stl.mean(axis=1).values, INDICE_535.values)[0, 1])

## 8. Validaciones V2 a V4

- **V2 · identidad.** $T + S + R$ debe reproducir la serie original. En `statsmodels` el residuo se define como lo que sobra, así que casi siempre se cumple; solo detecta errores de código, como fechas desalineadas.
- **V3 · residuo sin estructura.** Si STL capturó la tendencia y la estacionalidad, el residuo no debe mostrar patrón. Se revisa su autocorrelación a 1 y a 12 meses; con $n = 57$ la banda aproximada de ruido es $\pm 1.96/\sqrt{n} \approx \pm 0.26$. Un valor fuera de la banda indica estructura que STL no capturó (no invalida el resultado, pero se reporta).
- **V4 · sensibilidad.** Se repite la descomposición con `seasonal = 3` (patrón que puede cambiar más) y sin modo robusto. Si el mes pico casi no cambia, el resultado no depende del ajuste.

  *Cómo leerla:* con `seasonal = 3` el componente estacional tiene tanta libertad que puede absorber parte del ruido, y la fuerza estacional sube de forma artificial (es esperado). Por eso la comparación útil es el **mes pico** y la **correlación del perfil** con el principal, no el valor de $F_S$. Si el mes pico cambia entre variantes, el patrón estacional de ese producto no es estable con solo 4 o 5 observaciones por mes y debe tomarse con cautela. Un resultado así no es un error: se reporta tal cual.

In [ ]:
n = len(meses)
banda = 1.96 / np.sqrt(n)

# V2
dif = {c: float((desc[c]["observado"] - (desc[c]["tendencia"] + desc[c]["estacional"] + desc[c]["residuo"])).abs().max()) for c in claves_top}

# V3
v3 = pd.DataFrame({
    "autocorr. rezago 1": [acf(desc[c]["residuo"], nlags=12)[1] for c in claves_top],
    "autocorr. rezago 12": [acf(desc[c]["residuo"], nlags=12)[12] for c in claves_top],
}, index=claves_top)
v3["fuera de la banda"] = (v3.abs() > banda).any(axis=1)

# V4
res4 = {}
for nombre, params in VARIANTES.items():
    dv = {c: descomponer(matriz[c].astype(float), **params) for c in claves_top}
    res4[nombre] = {
        "Fs": {c: fuerza(dv[c]["estacional"], dv[c]["residuo"]) for c in claves_top},
        "pico": {c: MESES[perfil_estacional(dv[c]).idxmax() - 1] for c in claves_top},
        "corr_perfil": {c: np.corrcoef(perfil_estacional(dv[c]).values, perfiles[c].values)[0, 1] for c in claves_top},
    }
principal = list(VARIANTES)[0]
columnas = {}
for nombre in list(VARIANTES)[1:]:
    columnas[(nombre, "F_estacional")] = pd.Series(res4[nombre]["Fs"])
    columnas[(nombre, "mes pico")] = pd.Series(res4[nombre]["pico"])
    columnas[(nombre, "corr. del perfil con el principal")] = pd.Series(res4[nombre]["corr_perfil"])
    columnas[(nombre, "¿mismo mes pico?")] = pd.Series(res4[nombre]["pico"]) == metricas["mes_pico"]
v4 = pd.DataFrame(columnas).loc[claves_top]
v4.insert(0, ("principal", "F_estacional"), metricas["F_estacional"])
v4.insert(1, ("principal", "mes pico"), metricas["mes_pico"])
estable = v4.xs("¿mismo mes pico?", axis=1, level=1).all(axis=1)

validaciones["V1 integridad de los datos"] = "Sí"
validaciones["V2 identidad T + S + R"] = "Sí" if max(dif.values()) < 1e-6 else "No"
validaciones["V3 residuo sin estructura (productos dentro de la banda)"] = f"{int((~v3['fuera de la banda']).sum())} de {TOP_N}"
validaciones["V4 mismo mes pico en las dos variantes (productos)"] = f"{int(estable.sum())} de {TOP_N}"

print("V2: máxima diferencia con la serie original (piezas):", f"{max(dif.values()):.2e}")
print(f"\nV3: banda aproximada de ruido = ±{banda:.2f}")
display(v3.style.format({"autocorr. rezago 1": "{:.2f}", "autocorr. rezago 12": "{:.2f}"}))
print("V4: sensibilidad (se compara con la descomposición principal)")
display(v4.style.format({c: "{:.2f}" for c in v4.columns if c[1] in ("F_estacional", "corr. del perfil con el principal")}))

## 9. Aditivo frente a logaritmos

En un modelo **aditivo** el efecto de la temporada es una cantidad fija de piezas; en uno **multiplicativo** (descomposición de los logaritmos) es un porcentaje. Si la variación de un producto crece cuando su nivel crece, el multiplicativo es más adecuado y conviene modelar el logaritmo de las ventas.

**Cómo se compara.** Para cada producto se mide la correlación entre el tamaño del residuo ($|R|$) y el nivel de la tendencia. Si es cercana a 0, el residuo es parejo a lo largo del nivel (la versión describe bien la serie); si es positiva y alta, el residuo crece con el nivel. Se comparan ambas versiones y se registra cuál deja un residuo más parejo. La decisión de transformar o no la serie para SARIMA se toma en 5.5.

In [ ]:
filas = []
for c in claves_top:
    serie = matriz[c].astype(float)
    if (serie <= 0).any():
        filas.append({"clave": c, "nota": "tiene ceros: no se puede aplicar logaritmo"})
        continue
    d_log = descomponer(np.log(serie))
    d_ad = desc[c]
    corr_ad = np.corrcoef(d_ad["residuo"].abs(), d_ad["tendencia"])[0, 1]
    corr_log = np.corrcoef(d_log["residuo"].abs(), d_log["tendencia"])[0, 1]
    filas.append({
        "clave": c,
        "F_estacional aditivo": fuerza(d_ad["estacional"], d_ad["residuo"]),
        "F_estacional logaritmos": fuerza(d_log["estacional"], d_log["residuo"]),
        "corr(|residuo|, nivel) aditivo": corr_ad,
        "corr(|residuo|, nivel) logaritmos": corr_log,
        "residuo más parejo": "aditivo" if abs(corr_ad) <= abs(corr_log) else "logaritmos",
    })
comp_log = pd.DataFrame(filas).set_index("clave")
display(comp_log.style.format({c: "{:.2f}" for c in comp_log.columns if comp_log[c].dtype.kind == "f"}, na_rep="–"))
print(comp_log["residuo más parejo"].value_counts().to_string() if "residuo más parejo" in comp_log else "")

## 10. Atípicos: STL frente a Tukey (5.3.4)

Se marcan los meses atípicos con dos criterios, ambos solo hacia arriba (como en 5.3.4):
- **Tukey (5.3.4):** la venta del mes supera $Q_3 + 3 \times RIC$, con los cuartiles calculados solo sobre los meses con venta (productos con al menos 8 meses con venta).
- **STL:** el **residuo** del mes supera $Q_3 + 3 \times RIC$ calculado sobre los residuos del producto. Mide cuánto se aleja el mes de lo que esperaban la tendencia y la temporada, no de las ventas en bruto.

Como el segundo criterio descuenta la temporada, marca más meses: un mes puede ser normal en diciembre y raro en septiembre. Por eso se resume por producto y solo se detallan los meses que marca Tukey (los atípicos de referencia de 5.3.4), indicando si STL también los marca. Como el modelado será sobre estos 10 productos, los meses marcados son la lista que habrá que tratar al entrenar (5.5).

In [ ]:
def limite_tukey(valores, k=3.0):
    q1, q3 = valores.quantile([0.25, 0.75])
    return q3 + k * (q3 - q1)


def atipicos_tukey(serie, min_meses=8):
    """Criterio de 5.3.4: Q3 + 3×RIC sobre los meses con venta."""
    con_venta = serie[serie > 0]
    if len(con_venta) < min_meses:
        return pd.Series(False, index=serie.index)
    return serie > limite_tukey(con_venta)


def atipicos_stl(residuo):
    """Mismo criterio aplicado al residuo de STL (descuenta tendencia y temporada)."""
    return residuo > limite_tukey(residuo)


resumen, detalle = [], []
for c in claves_top:
    d = desc[c]
    t, s = atipicos_tukey(matriz[c].astype(float)), atipicos_stl(d["residuo"])
    resumen.append({"clave": c, "meses marcados por Tukey": int(t.sum()), "meses marcados por STL": int(s.sum()),
                    "marcados por ambos": int((t & s).sum())})
    for f in d.index[t]:
        detalle.append({"clave": c, "mes": f.strftime("%Y-%m"), "piezas": d.loc[f, "observado"],
                        "residuo STL": d.loc[f, "residuo"], "¿también lo marca STL?": "sí" if s[f] else "no",
                        "pedido confirmado": "sí" if (c, f.strftime("%Y-%m-01")) == PEDIDO_CONFIRMADO else ""})
resumen = pd.DataFrame(resumen).set_index("clave")
display(resumen)
print(f"Total de meses: {len(meses) * TOP_N}. Marcados por Tukey: {resumen.iloc[:, 0].sum()} · por STL: {resumen.iloc[:, 1].sum()} · por ambos: {resumen.iloc[:, 2].sum()}")
if detalle:
    print("\nMeses marcados por Tukey:")
    display(pd.DataFrame(detalle).style.format({"piezas": "{:,.0f}", "residuo STL": "{:+,.0f}"}).hide(axis="index"))

## 11. Resumen y decisiones para 5.5

La tabla siguiente reúne el resultado de las validaciones y los indicadores principales. Las decisiones de modelado **no se toman aquí**: se registran para 5.5 una vez revisados los resultados.

In [ ]:
display(pd.Series(validaciones, name="resultado").to_frame())
print(f"Fuerza de tendencia (mediana de los {TOP_N}): {metricas['F_tendencia'].median():.2f}")
print(f"Fuerza estacional (mediana de los {TOP_N}): {metricas['F_estacional'].median():.2f}")
print(f"Productos con tendencia a la baja en los últimos 12 meses: {(metricas['tendencia_12m_%'] < 0).sum()} de {TOP_N}")
print("Mes pico más frecuente:", metricas["mes_pico"].mode().iloc[0])

## Lectura de resultados y decisiones para 5.5

### Lo observado en las gráficas
- **Tendencia.** Se distingue en los 10 productos, pero no es una línea recta: cambia de dirección durante el periodo. En 7 de 10 baja en los últimos 12 meses (de −4% a −50%). Solo suben el 1014 (+16%), el 8311 (+7%) y el 11082 (+6%).
- **Estacionalidad.** Hay picos y valles cada año (el mes pico queda de +38% a +78% sobre la tendencia y el valle de −27% a −61%), pero el patrón no es limpio. El mes pico cambia al mover los ajustes en 7 de 10 productos.
- **Ruido.** Es considerable: la fuerza estacional mediana es 0.15 y la de tendencia 0.23. Hay picos aislados, como los tres sheet masks en octubre de 2023. Las causas probables son promociones y pedidos grandes, pero los datos no permiten comprobarlo.
- **Productos con señal más clara:** 1014, 11082, 731511, 731611 y 11011. **Los más ruidosos:** los tres sheet masks (919411, 919211, 919111), con fuerza de tendencia y estacional cercana a 0, y también 8811 y 8311.

### Respuestas a las preguntas

**1. ¿La estacionalidad es lo bastante marcada para incluir componente estacional en SARIMA?** Solo en parte. La fuerza estacional es de 0.24 a 0.38 en cuatro productos (1014, 11011, 11082, 731511) y de 0.17 o menos en los otros seis. El mes pico se mantiene al mover los ajustes solo en 11082 (dic), 1014 (dic) y 731511 (nov). En conjunto, el patrón de los 10 se parece al índice de 5.3.5 (correlación 0.64), pero producto por producto la coincidencia es débil. Con solo 4 o 5 observaciones por mes, el patrón de cada producto es incierto. En 5.5 se prueba cada producto con y sin componente estacional y se elige por holdout, sin asumirlo.

**2. ¿La tendencia de 2026 exige diferenciación?** STL muestra cambios graduales de nivel, no un salto en 2026. En varios productos la baja empieza en 2025 (sheet masks, 731511, 731611, 8811). En 919411 y 919111 la tendencia baja, sube y vuelve a bajar, así que no hay un nivel fijo alrededor del cual oscile la serie. La fuerza de tendencia solo pasa de 0.5 en 1014, 731511 y 731611. En los sheet masks casi no se distingue del ruido (≤0.02), así que ahí la gráfica no decide. En 5.5 se compara d = 0 contra d = 1 por producto. La tendencia de STL es menos confiable en los últimos meses, por lo que no se debe sobreinterpretar.

**3. ¿Conviene modelar el logaritmo de las ventas?** No hay ganador claro. El residuo más parejo sale con la versión aditiva en 5 productos y con la de logaritmos en otros 5. En 731511 y 731611 el residuo crece con el nivel en ambas versiones (correlación de 0.38 a 0.44) y el logaritmo no lo corrige. El logaritmo sube la fuerza estacional en 7 de 10, pero eso se debe en parte a que comprime los picos y no prueba un mejor pronóstico. Se mantiene la versión aditiva como principal. El logaritmo se prueba como alternativa en 5.5 (todos los productos vendieron en los 57 meses, así que se puede aplicar).

**4. ¿Qué meses se tratan como atípicos al entrenar?** Tukey marca 4 meses:
- 2023-10, en los tres sheet masks (919411: 1,250 piezas; 919211: 1,176; 919111: 863).
- 2024-03, en 919411 (983 piezas).

Que los tres coincidan en el mismo mes apunta a un evento común, como un pedido grande o una promoción. Hay que confirmarlo con la empresa, igual que el kit 8419 de 2022-12. Esos 4 meses son los que se tratan como atípicos, con el umbral calculado solo sobre datos de entrenamiento. STL marca 49 de 570 meses (8.6%) y los 4 de Tukey están entre ellos. Los otros 45 no se eliminan automáticamente. En 11082 y 1014 el residuo se mantiene alto varios meses seguidos (autocorrelación de 0.47 y 0.50, fuera de la banda), lo que apunta a episodios y no a picos aislados. Son candidatos para revisar con la empresa.

**5. ¿Algún producto del top 10 ya no es representativo?** Ninguno ha desaparecido. Vendieron en los 57 meses y todos siguen dentro de los 19 primeros lugares de 352 en los últimos 12 meses. Pero 5 de 10 ya no están en el top 10 reciente (8811 lugar 18, 8311 lugar 19, 731511 lugar 13, 11011 lugar 15, 731611 lugar 11), y los 10 pasaron de 20.0% de las piezas en todo el historial a 16.9% en los últimos 12 meses. Se mantiene el criterio histórico definido con el asesor y se documenta esta salvedad en 5.3.7 y 5.5.

### ¿Se recomienda seguir con el modelado?
**Sí, con expectativas realistas.** Los datos son completos y consistentes. Hay señal en tendencia, y estacionalidad en al menos cuatro productos. Además, el objetivo del proyecto es comparar contra el modelo base, y que un modelo no lo supere en un producto ruidoso también es un resultado válido y reportable. El nivel de ruido hace posible que los umbrales de éxito (MAPE ≤ 20% y −15% frente al base) no se cumplan en los productos más ruidosos (sheet masks, 8811, 8311). Se reporta tal cual, sin ajustar los umbrales después.

**Para 5.5:**
1. Calcular primero el modelo base y fijar el umbral de WAPE antes de entrenar candidatos.
2. Probar cada producto con y sin estacionalidad, y con d = 0 contra d = 1.
3. Probar el logaritmo como alternativa.
4. Tratar como atípicos los 4 meses de Tukey y confirmar con la empresa los eventos de octubre de 2023 y marzo de 2024.
5. Incluir el mes como variable de calendario en los modelos que lo admitan.